# Treinamento YOLO11 - Tracking de Jogadores e Bola

Este notebook foi feito para rodar no **Google Colab**.
Certifique-se de alterar o ambiente de execução para usar uma GPU: `Runtime > Change runtime type > Hardware accelerator > T4 GPU` (ou superior).

In [ ]:
!nvidia-smi

Sat Jun 27 19:13:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Instalação das Bibliotecas

Vamos instalar o YOLO (Ultralytics) e a biblioteca do Roboflow para fazer o download automático do dataset que você escolheu.

In [ ]:
!pip install ultralytics roboflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 148.3 MB/s eta 0:00:00


## 2. Download do Dataset (Roboflow)

No site do Roboflow Universe, vá até o dataset escolhido, clique em **Download Dataset**, selecione o formato **YOLO11** e copie o código. Cole ele na célula abaixo:

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="U5pE9XdhmIXd92M87wY8")
project = rf.workspace("roboflow-universe-projects").project("soccer-players-ckbru")
version = project.version(15)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Soccer-Players-15 in yolov11:: 100%|██████████| 2166/2166 [00:00<00:00, 9239.89it/s]


## 3. Treinamento do Modelo

Usaremos a versão `yolo11n` (nano) ou `yolo11s` (small) para que rode rápido e de forma otimizada para a sua placa local (GTX 1650) posteriormente.

In [ ]:
from ultralytics import YOLO

# Carrega um modelo pré-treinado na base COCO (para acelerar o aprendizado)
model = YOLO('yolo11s.pt')

# Inicia o treinamento
# ATENÇÃO: mude 'dataset.location' se precisar ajustar o caminho do arquivo data.yaml baixado.
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=500,             # Número de vezes que o modelo verá todo o dataset
    imgsz=640,             # Resolução base
    batch=16,              # Imagens processadas de uma vez
    name='yolo_football',  # Nome do experimento
    device=0               # Usar GPU (0)
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Soccer-Players-15/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, h

## 4. Exportação

Após o treinamento, o melhor modelo ficará salvo na pasta `runs/detect/yolo_football/weights/best.pt`.

Você precisará **baixar esse arquivo `best.pt`** para a sua máquina local (salve dentro da pasta `weights/` do nosso projeto).

In [ ]:
import shutil
from google.colab import files

# Baixando o modelo automaticamente no final do treino
files.download('runs/detect/yolo_football/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>